In [1]:
# Load environment variables from the .env file created by 00-Local-Setup.ipynb
%reload_ext dotenv
%dotenv ../.env

import os
from pathlib import Path

from graphrag_toolkit.lexical_graph import LexicalGraphIndex, set_logging_config
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from graphrag_toolkit.lexical_graph.storage import VectorStoreFactory
from graphrag_toolkit.lexical_graph.indexing.load import FileBasedDocs
from graphrag_toolkit.lexical_graph.indexing.build import Checkpoint

from llama_index.readers.web import SimpleWebPageReader

# Set logging level
set_logging_config('INFO')

# Verify environment variables are loaded
print("Environment Configuration:")
print(f"  AWS Region: {os.environ.get('AWS_REGION')}")
print(f"  Graph Store: {os.environ.get('GRAPH_STORE')}")
print(f"  Vector Store: {os.environ.get('VECTOR_STORE')}")
print(f"  Extraction Model: {os.environ.get('EXTRACTION_MODEL')}")
print(f"  Embeddings Model: {os.environ.get('EMBEDDINGS_MODEL')}")
print(f"  Embeddings Dimensions: {os.environ.get('EMBEDDINGS_DIMENSIONS')}")

# Verify dimensions match
expected_dims = "1024"
actual_dims = os.environ.get('EMBEDDINGS_DIMENSIONS')
if actual_dims != expected_dims:
    print(f"\n⚠️  WARNING: EMBEDDINGS_DIMENSIONS is {actual_dims}, but should be {expected_dims} for Titan Embed Text v2")
    print("   Please update your .env file and restart the notebook.")
else:
    print(f"\n✅ Embeddings dimensions correctly configured: {actual_dims}")

# Create extracted directory if it doesn't exist
extracted_dir = Path('../extracted')
extracted_dir.mkdir(exist_ok=True)

# Initialize document storage and checkpoint
extracted_docs = FileBasedDocs(
    docs_directory='../extracted'
)

checkpoint = Checkpoint('../extraction-checkpoint')

# Initialize graph and vector stores using environment variables
graph_store = GraphStoreFactory.for_graph_store(os.environ['GRAPH_STORE'])
vector_store = VectorStoreFactory.for_vector_store(os.environ['VECTOR_STORE'])

# Create the lexical graph index
graph_index = LexicalGraphIndex(
    graph_store, 
    vector_store
)

print("\n✅ Setup complete - ready for extraction!")


Environment Configuration:
  AWS Region: us-east-1
  Graph Store: neptune-graph://g-4o5cjg3ix7
  Vector Store: aoss://https://vgoh4hviohrcmbum3617.us-east-1.aoss.amazonaws.com
  Extraction Model: us.anthropic.claude-3-haiku-20240307-v1:0
  Embeddings Model: amazon.titan-embed-text-v2:0
  Embeddings Dimensions: 1024

✅ Embeddings dimensions correctly configured: 1024

✅ Setup complete - ready for extraction!


In [2]:
# Sample AWS Neptune documentation URLs
doc_urls = [
    'https://docs.aws.amazon.com/neptune-analytics/latest/userguide/what-is-neptune-analytics.html',
    'https://docs.aws.amazon.com/neptune-analytics/latest/userguide/neptune-analytics-vs-neptune-database.html'
]

print("Loading documents from AWS Neptune documentation...")
print(f"Documents to process: {len(doc_urls)}")

# Load documents using SimpleWebPageReader
docs = SimpleWebPageReader(
    html_to_text=True,
    metadata_fn=lambda url: {'url': url, 'source': 'aws-neptune-docs'}
).load_data(doc_urls)

print(f"\n✅ Loaded {len(docs)} documents successfully")
for i, doc in enumerate(docs):
    print(f"  Document {i+1}: {doc.metadata.get('url', 'Unknown URL')}")
    print(f"    Length: {len(doc.text)} characters")


Loading documents from AWS Neptune documentation...
Documents to process: 2

✅ Loaded 2 documents successfully
  Document 1: https://docs.aws.amazon.com/neptune-analytics/latest/userguide/what-is-neptune-analytics.html
    Length: 2292 characters
  Document 2: https://docs.aws.amazon.com/neptune-analytics/latest/userguide/neptune-analytics-vs-neptune-database.html
    Length: 2730 characters


In [3]:
print("Starting extraction process...")
print("This may take several minutes depending on document size and model performance.")
print("\nUsing models:")
print(f"  - Extraction: {os.environ.get('EXTRACTION_MODEL')}")
print(f"  - Embeddings: {os.environ.get('EMBEDDINGS_MODEL')} ({os.environ.get('EMBEDDINGS_DIMENSIONS')} dimensions)")

# Run the extraction process
graph_index.extract(
    docs, 
    handler=extracted_docs, 
    checkpoint=checkpoint, 
    show_progress=True
)

# Get the collection ID for the next stage
collection_id = extracted_docs.collection_id

print('\n✅ Extraction complete!')
print(f'Collection ID: {collection_id}')
print(f'Extracted files saved to: {extracted_dir.absolute()}')

# List extracted files
extracted_files = list(extracted_dir.glob('*'))
if extracted_files:
    print(f'\nExtracted files ({len(extracted_files)}):')
    for file in extracted_files[:10]:  # Show first 10 files
        print(f'  - {file.name}')
    if len(extracted_files) > 10:
        print(f'  ... and {len(extracted_files) - 10} more files')


Starting extraction process...
This may take several minutes depending on document size and model performance.

Using models:
  - Extraction: us.anthropic.claude-3-haiku-20240307-v1:0
  - Embeddings: amazon.titan-embed-text-v2:0 (1024 dimensions)
2025-07-06 21:15:49:INFO:g.l.i.e.extraction_pipeline:Running extraction pipeline [batch_size: 4, num_workers: 2]


Extracting propositions [nodes: 3, num_workers: 4]: 100%|██████████| 3/3 [00:02<00:00,  1.50it/s]
Extracting propositions [nodes: 3, num_workers: 4]: 100%|██████████| 3/3 [00:02<00:00,  1.01it/s]
Extracting topics [nodes: 3, num_workers: 4]: 100%|██████████| 3/3 [00:08<00:00,  2.97s/it]


2025-07-06 21:16:04:INFO:g.l.i.b.build_pipeline:Running build pipeline [batch_size: 4, num_workers: 1, job_sizes: [142], batch_writes_enabled: True, batch_write_size: 25]

✅ Extraction complete!
Collection ID: 20250706-211532
Extracted files saved to: /Users/manojs/Documents/Code/graphrag-toolkit/examples/lexical-graph/notebooks/../extracted

Extracted files (2):
  - 20250706-211532
  - 20250706-211439


In [4]:
# # Fix OpenSearch index dimension mismatch (no extra modules needed)
# import boto3
# import os
# import json
# import time

# def fix_opensearch_collection():
#     """Recreate the OpenSearch collection to fix dimension issues."""
#     try:
#         print("🔧 Fixing OpenSearch collection dimension mismatch...")
        
#         # Initialize OpenSearch client
#         opensearch = boto3.client('opensearchserverless', region_name='us-east-1')
        
#         # Get collection details
#         collection_name = "graphrag-local-collection"
        
#         try:
#             # Get existing collection
#             response = opensearch.batch_get_collection(names=[collection_name])
#             collection_id = response['collectionDetails'][0]['id']
#             print(f"📋 Found existing collection: {collection_id}")
            
#             # Delete the collection
#             print(f"🗑️  Deleting collection to fix dimension issues...")
#             opensearch.delete_collection(id=collection_id)
#             print(f"✅ Successfully deleted collection")
            
#             # Wait for deletion to complete
#             print("⏳ Waiting for deletion to complete...")
#             time.sleep(30)
            
#         except Exception as e:
#             print(f"ℹ️  Collection not found or already deleted: {e}")
        
#         # Recreate the collection
#         print("🔄 Recreating collection with correct configuration...")
#         response = opensearch.create_collection(
#             name=collection_name,
#             type="VECTORSEARCH",
#             standbyReplicas="DISABLED"
#         )
        
#         new_collection_id = response['createCollectionDetail']['id']
#         print(f"✅ Successfully recreated collection: {new_collection_id}")
        
#         # Wait for collection to be active
#         print("⏳ Waiting for collection to become active...")
#         while True:
#             status_response = opensearch.batch_get_collection(names=[collection_name])
#             status = status_response['collectionDetails'][0]['status']
#             print(f"   Collection status: {status}")
#             if status == "ACTIVE":
#                 break
#             time.sleep(10)
        
#         print(f"\n✅ OpenSearch collection fix complete!")
#         print(f"💡 Next steps:")
#         print(f"   1. Update your .env file with the new collection endpoint")
#         print(f"   2. Re-run your notebook")
#         print(f"   3. The new collection will use correct 1024 dimensions")
        
#         # Show the new endpoint
#         new_endpoint = response['createCollectionDetail']['collectionEndpoint']
#         print(f"\n📝 New collection endpoint: {new_endpoint}")
#         print(f"   Update your .env file VECTOR_STORE to: aoss://{new_endpoint}")
        
#     except Exception as e:
#         print(f"❌ Error fixing OpenSearch collection: {e}")
#         print(f"\nManual steps:")
#         print(f"1. Go to AWS Console > OpenSearch Serverless")
#         print(f"2. Delete the 'graphrag-local-collection'")
#         print(f"3. Recreate it with the same settings")
#         print(f"4. Update your .env file with the new endpoint")

# # Run the fix
# fix_opensearch_collection()

In [5]:
# Reload environment variables (in case running in a new session)
%reload_ext dotenv
%dotenv ../.env

import os
from pathlib import Path

from graphrag_toolkit.lexical_graph import LexicalGraphIndex, set_logging_config
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from graphrag_toolkit.lexical_graph.storage import VectorStoreFactory
from graphrag_toolkit.lexical_graph.indexing.load import FileBasedDocs
from graphrag_toolkit.lexical_graph.indexing.build import Checkpoint

# Set logging level
set_logging_config('INFO')

print("Preparing for build stage...")
print(f"Using collection ID: {collection_id}")

# Verify dimensions are still correct
expected_dims = "1024"
actual_dims = os.environ.get('EMBEDDINGS_DIMENSIONS')
if actual_dims != expected_dims:
    print(f"\n⚠️  WARNING: EMBEDDINGS_DIMENSIONS is {actual_dims}, but should be {expected_dims}")
    print("   This may cause indexing errors. Please update your .env file.")
else:
    print(f"\n✅ Embeddings dimensions correctly configured: {actual_dims}")

# Load the extracted documents
docs = FileBasedDocs(
    docs_directory='../extracted',
    collection_id=collection_id
)

# Create build checkpoint
checkpoint = Checkpoint('../build-checkpoint')

# Initialize stores (same as extraction)
graph_store = GraphStoreFactory.for_graph_store(os.environ['GRAPH_STORE'])
vector_store = VectorStoreFactory.for_vector_store(os.environ['VECTOR_STORE'])

# Create the lexical graph index
graph_index = LexicalGraphIndex(
    graph_store, 
    vector_store
)

print("\nStarting build process...")
print("This will create the knowledge graph and vector embeddings.")
print("\nTarget stores:")
print(f"  - Graph Store (Neptune): {os.environ.get('GRAPH_STORE')}")
print(f"  - Vector Store (OpenSearch): {os.environ.get('VECTOR_STORE')}")

# Run the build process
graph_index.build(
    docs, 
    checkpoint=checkpoint, 
    show_progress=True
)

print('\n✅ Build complete!')
print('\nYour GraphRAG knowledge graph is now ready for querying!')
print('\nNext steps:')
print('  1. Use the 03-Traversal-Based-Querying.ipynb notebook for graph traversal queries')
print('  2. Use the 04-Semantic-Guided-Querying.ipynb notebook for semantic search')
print('  3. Use the 06-Agentic-GraphRAG.ipynb notebook for agentic workflows')


Preparing for build stage...
Using collection ID: 20250706-211532

✅ Embeddings dimensions correctly configured: 1024

Starting build process...
This will create the knowledge graph and vector embeddings.

Target stores:
  - Graph Store (Neptune): neptune-graph://g-4o5cjg3ix7
  - Vector Store (OpenSearch): aoss://https://vgoh4hviohrcmbum3617.us-east-1.aoss.amazonaws.com
2025-07-06 21:16:25:INFO:g.l.i.b.build_pipeline:Running build pipeline [batch_size: 4, num_workers: 2, job_sizes: [90, 52], batch_writes_enabled: True, batch_write_size: 25]


Building graph [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 52/52 [00:00<00:00, 56401.30it/s]
Building graph [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 90/90 [00:00<00:00, 47239.06it/s]
[00cc0/*] Retrying query in 5.421641495862817 seconds because it raised ConflictException: An error occurred (ConflictException) when calling the ExecuteQuery operation (reached max retries: 0): Operation failed due to conflicting concurrent operations (please retry), 0 transactions are currently rolling back. [attempt: 1, query: **REDACTED**, parameters: **REDACTED**]
Building vector index [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 52/52 [00:00<00:00, 396552.38it/s]
Building vector index [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 90/90 [00:00<00:00, 18255.51it/s]



✅ Build complete!

Your GraphRAG knowledge graph is now ready for querying!

Next steps:
  1. Use the 03-Traversal-Based-Querying.ipynb notebook for graph traversal queries
  2. Use the 04-Semantic-Guided-Querying.ipynb notebook for semantic search
  3. Use the 06-Agentic-GraphRAG.ipynb notebook for agentic workflows


In [ ]:
print("Verifying data in stores...")

try:
    # Check graph store
    print("\n📊 Graph Store (Neptune Analytics):")
    # You can add specific graph queries here to verify data
    print("  ✅ Connected to Neptune Analytics Graph")
    print(f"  Graph ID: {os.environ.get('GRAPH_STORE', '').split('//')[-1]}")
    
    # Check vector store
    print("\n🔍 Vector Store (OpenSearch Serverless):")
    print("  ✅ Connected to OpenSearch Serverless Collection")
    collection_endpoint = os.environ.get('VECTOR_STORE', '').split('//')[-1]
    print(f"  Collection Endpoint: {collection_endpoint}")
    
    # Verify dimensions
    print("\n🎯 Configuration Verification:")
    print(f"  Embeddings Model: {os.environ.get('EMBEDDINGS_MODEL')}")
    print(f"  Embeddings Dimensions: {os.environ.get('EMBEDDINGS_DIMENSIONS')}")
    print(f"  Neptune Graph Vector Dimensions: 1024 (configured)")
    
    if os.environ.get('EMBEDDINGS_DIMENSIONS') == '1024':
        print("  ✅ Dimension configuration is correct!")
    else:
        print("  ⚠️  Dimension mismatch detected!")
    
    print("\n✅ Verification complete - both stores are accessible!")
    
except Exception as e:
    print(f"\n❌ Verification failed: {e}")
    print("Please check your AWS credentials and resource configurations.")


# 02 - Separate Extract and Build (Local Setup)

This notebook demonstrates how to run the extract and build stages separately using the local GraphRAG environment setup from the `00-Local-Setup.ipynb` notebook.

## Setup

**Prerequisites:** Make sure you have completed the [Local Setup](./00-Local-Setup.ipynb) notebook first to create:
- Neptune Analytics Graph
- OpenSearch Serverless Collection
- `.env` file with your AWS resources and model configurations
- `extracted` directory for intermediate files

## Extract

The extract stage processes documents and extracts entities, relationships, and communities using the configured models.

In [1]:
# Load environment variables from the .env file created by 00-Local-Setup.ipynb
%reload_ext dotenv
%dotenv ../.env

import os
from pathlib import Path

from graphrag_toolkit.lexical_graph import LexicalGraphIndex, set_logging_config
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from graphrag_toolkit.lexical_graph.storage import VectorStoreFactory
from graphrag_toolkit.lexical_graph.indexing.load import FileBasedDocs
from graphrag_toolkit.lexical_graph.indexing.build import Checkpoint

from llama_index.readers.web import SimpleWebPageReader

# Set logging level
set_logging_config('INFO')

# Verify environment variables are loaded
print("Environment Configuration:")
print(f"  AWS Region: {os.environ.get('AWS_REGION')}")
print(f"  Graph Store: {os.environ.get('GRAPH_STORE')}")
print(f"  Vector Store: {os.environ.get('VECTOR_STORE')}")
print(f"  Extraction Model: {os.environ.get('EXTRACTION_MODEL')}")
print(f"  Embeddings Model: {os.environ.get('EMBEDDINGS_MODEL')}")

# Create extracted directory if it doesn't exist
extracted_dir = Path('../extracted')
extracted_dir.mkdir(exist_ok=True)

# Initialize document storage and checkpoint
extracted_docs = FileBasedDocs(
    docs_directory='../extracted'
)

checkpoint = Checkpoint('../extraction-checkpoint')

# Initialize graph and vector stores using environment variables
graph_store = GraphStoreFactory.for_graph_store(os.environ['GRAPH_STORE'])
vector_store = VectorStoreFactory.for_vector_store(os.environ['VECTOR_STORE'])

# Create the lexical graph index
graph_index = LexicalGraphIndex(
    graph_store, 
    vector_store
)

print("\n✅ Setup complete - ready for extraction!")

Environment Configuration:
  AWS Region: us-east-1
  Graph Store: neptune-graph://g-5z6r78ydka
  Vector Store: aoss://https://79ld1ovkwb35j2v485if.us-east-1.aoss.amazonaws.com
  Extraction Model: us.anthropic.claude-3-haiku-20240307-v1:0
  Embeddings Model: amazon.titan-embed-text-v2:0

✅ Setup complete - ready for extraction!


## Load Sample Documents

We'll use AWS Neptune documentation as sample data for the GraphRAG extraction.


In [2]:
# Sample AWS Neptune documentation URLs
doc_urls = [
    'https://docs.aws.amazon.com/neptune-analytics/latest/userguide/what-is-neptune-analytics.html',
    'https://docs.aws.amazon.com/neptune-analytics/latest/userguide/neptune-analytics-vs-neptune-database.html'
]

print("Loading documents from AWS Neptune documentation...")
print(f"Documents to process: {len(doc_urls)}")

# Load documents using SimpleWebPageReader
docs = SimpleWebPageReader(
    html_to_text=True,
    metadata_fn=lambda url: {'url': url, 'source': 'aws-neptune-docs'}
).load_data(doc_urls)

print(f"\n✅ Loaded {len(docs)} documents successfully")
for i, doc in enumerate(docs):
    print(f"  Document {i+1}: {doc.metadata.get('url', 'Unknown URL')}")
    print(f"    Length: {len(doc.text)} characters")


Loading documents from AWS Neptune documentation...
Documents to process: 2

✅ Loaded 2 documents successfully
  Document 1: https://docs.aws.amazon.com/neptune-analytics/latest/userguide/what-is-neptune-analytics.html
    Length: 2292 characters
  Document 2: https://docs.aws.amazon.com/neptune-analytics/latest/userguide/neptune-analytics-vs-neptune-database.html
    Length: 2730 characters


## Run Extraction

This will extract entities, relationships, and communities from the documents using the models configured in your `.env` file.


In [3]:
print("Starting extraction process...")
print("This may take several minutes depending on document size and model performance.")
print("\nUsing models:")
print(f"  - Extraction: {os.environ.get('EXTRACTION_MODEL')}")
print(f"  - Embeddings: {os.environ.get('EMBEDDINGS_MODEL')}")

# Run the extraction process
graph_index.extract(
    docs, 
    handler=extracted_docs, 
    checkpoint=checkpoint, 
    show_progress=True
)

# Get the collection ID for the next stage
collection_id = extracted_docs.collection_id

print('\n✅ Extraction complete!')
print(f'Collection ID: {collection_id}')
print(f'Extracted files saved to: {extracted_dir.absolute()}')

# List extracted files
extracted_files = list(extracted_dir.glob('*'))
if extracted_files:
    print(f'\nExtracted files ({len(extracted_files)}):')
    for file in extracted_files[:10]:  # Show first 10 files
        print(f'  - {file.name}')
    if len(extracted_files) > 10:
        print(f'  ... and {len(extracted_files) - 10} more files')


Starting extraction process...
This may take several minutes depending on document size and model performance.

Using models:
  - Extraction: us.anthropic.claude-3-haiku-20240307-v1:0
  - Embeddings: amazon.titan-embed-text-v2:0
2025-07-06 18:46:03:INFO:g.l.i.e.extraction_pipeline:Running extraction pipeline [batch_size: 4, num_workers: 2]


Extracting propositions [nodes: 3, num_workers: 4]: 100%|██████████| 3/3 [00:01<00:00,  1.74it/s]
Extracting propositions [nodes: 3, num_workers: 4]: 100%|██████████| 3/3 [00:02<00:00,  1.09it/s]
Extracting topics [nodes: 3, num_workers: 4]: 100%|██████████| 3/3 [00:05<00:00,  1.93s/it]


2025-07-06 18:46:15:INFO:g.l.i.b.build_pipeline:Running build pipeline [batch_size: 4, num_workers: 1, job_sizes: [134], batch_writes_enabled: True, batch_write_size: 25]

✅ Extraction complete!
Collection ID: 20250706-184530
Extracted files saved to: /Users/manojs/Documents/Code/graphrag-toolkit/examples/lexical-graph/notebooks/../extracted

Extracted files (1):
  - 20250706-184530


## Build

The build stage takes the extracted data and creates the knowledge graph in Neptune Analytics and vector embeddings in OpenSearch Serverless.

In [ ]:
# Reload environment variables (in case running in a new session)
%reload_ext dotenv
%dotenv ../.env

import os
from pathlib import Path

from graphrag_toolkit.lexical_graph import LexicalGraphIndex, set_logging_config
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from graphrag_toolkit.lexical_graph.storage import VectorStoreFactory
from graphrag_toolkit.lexical_graph.indexing.load import FileBasedDocs
from graphrag_toolkit.lexical_graph.indexing.build import Checkpoint

# Set logging level
set_logging_config('INFO')

print("Preparing for build stage...")
print(f"Using collection ID: {collection_id}")

# Load the extracted documents
docs = FileBasedDocs(
    docs_directory='../extracted',
    collection_id=collection_id
)

# Create build checkpoint
checkpoint = Checkpoint('../build-checkpoint')

# Initialize stores (same as extraction)
graph_store = GraphStoreFactory.for_graph_store(os.environ['GRAPH_STORE'])
vector_store = VectorStoreFactory.for_vector_store(os.environ['VECTOR_STORE'])

# Create the lexical graph index
graph_index = LexicalGraphIndex(
    graph_store, 
    vector_store
)

print("\nStarting build process...")
print("This will create the knowledge graph and vector embeddings.")
print("\nTarget stores:")
print(f"  - Graph Store (Neptune): {os.environ.get('GRAPH_STORE')}")
print(f"  - Vector Store (OpenSearch): {os.environ.get('VECTOR_STORE')}")

# Run the build process
graph_index.build(
    docs, 
    checkpoint=checkpoint, 
    show_progress=True
)

print('\n✅ Build complete!')
print('\nYour GraphRAG knowledge graph is now ready for querying!')

Preparing for build stage...
Using collection ID: 20250706-184530

Starting build process...
This will create the knowledge graph and vector embeddings.

Target stores:
  - Graph Store (Neptune): neptune-graph://g-5z6r78ydka
  - Vector Store (OpenSearch): aoss://https://79ld1ovkwb35j2v485if.us-east-1.aoss.amazonaws.com
2025-07-06 18:46:34:INFO:g.l.i.b.build_pipeline:Running build pipeline [batch_size: 4, num_workers: 2, job_sizes: [75, 59], batch_writes_enabled: True, batch_write_size: 25]


Building graph [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 59/59 [00:00<00:00, 49246.55it/s]
Building graph [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 75/75 [00:00<00:00, 59850.23it/s]
[1a342/*] Retrying query in 1.2709368907504457 seconds because it raised ConflictException: An error occurred (ConflictException) when calling the ExecuteQuery operation (reached max retries: 0): Operation failed due to conflicting concurrent operations (please retry), 0 transactions are currently rolling back. [attempt: 1, query: **REDACTED**, parameters: **REDACTED**]
Building vector index [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 75/75 [00:00<00:00, 157838.84it/s]
Building vector index [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 59/59 [00:00<00:00, 342747.83it/s]


BulkIndexError: ('3 document(s) failed to index.', [{'index': {'_index': 'chunk', '_id': '1%3A0%3AcTOQ4pcBEKP59Kk1axA_', 'status': 400, 'error': {'type': 'mapper_parsing_exception', 'reason': "failed to parse field [embedding] of type [knn_vector] in document with id 'cTOQ4pcBEKP59Kk1axA_'. Preview of field's value: 'null'", 'caused_by': {'type': 'illegal_argument_exception', 'reason': 'Vector dimension mismatch. Expected: 1536, Given: 1024'}}, 'data': {'embedding': [-0.04216631129384041, -0.029189318418502808, 0.037564173340797424, 0.007671896368265152, -0.009397421032190323, -0.002852862933650613, 0.0037424995098263025, -0.06913919746875763, 0.06410446017980576, -0.013409235514700413, -0.0021848753094673157, -0.00801779329776764, 0.005900789052248001, -0.015106023289263248, -0.037946537137031555, -0.026607461273670197, 0.04312899708747864, -0.0003825174644589424, 0.003011266700923443, 0.05207638442516327, -0.00045944013982079923, 0.023733656853437424, 0.0890103206038475, -0.005262335

In [ ]:
print("Verifying data in stores...")

try:
    # Check graph store
    print("\n📊 Graph Store (Neptune Analytics):")
    # You can add specific graph queries here to verify data
    print("  ✅ Connected to Neptune Analytics Graph")
    print(f"  Graph ID: {os.environ.get('GRAPH_STORE', '').split('//')[-1]}")
    
    # Check vector store
    print("\n🔍 Vector Store (OpenSearch Serverless):")
    print("  ✅ Connected to OpenSearch Serverless Collection")
    collection_endpoint = os.environ.get('VECTOR_STORE', '').split('//')[-1]
    print(f"  Collection Endpoint: {collection_endpoint}")
    
    print("\n✅ Verification complete - both stores are accessible!")
    
except Exception as e:
    print(f"\n❌ Verification failed: {e}")
    print("Please check your AWS credentials and resource configurations.")
